# 04. FastConformerBPE — обучение BPE-vocab ASR

Self-contained ноутбук: от сырых данных до submission.
Архитектура: Fast Conformer с BPE-256 словарём (~4.2M параметров, subsample_factor=4).
HP Random Search оборачивает весь цикл обучения.

## Установка зависимостей

Выполнять под свою платформу — локально обычно уже установлено через `uv sync`; на Colab/Kaggle — раскомментируй нужный блок.

In [ ]:
# Idempotent data download. Skip if notebooks/data/ already populated.
from pathlib import Path
import zipfile

DATA_ROOT = Path("notebooks/data")
_ZIP_URL = "https://drive.google.com/file/d/1WOubhQ4LtPYEZTOHNkZiDqIobfOQEWBW/view?usp=share_link"
_ZIP_PATH = DATA_ROOT / "data.zip"

if not DATA_ROOT.exists() or not any(DATA_ROOT.iterdir()):
    DATA_ROOT.mkdir(parents=True, exist_ok=True)
    import gdown

    gdown.download(url=_ZIP_URL, output=str(_ZIP_PATH), quiet=False, fuzzy=True)
    with zipfile.ZipFile(_ZIP_PATH) as zf:
        zf.extractall(DATA_ROOT)
    print(f"Extracted data to {DATA_ROOT}")
else:
    print(f"Data already present at {DATA_ROOT} — skipping download.")


## Env hardening

In [ ]:
# Env hardening — must run BEFORE `import torch`.
import os

# Снижает фрагментацию VRAM — критично для torch.compile + переменных T.
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import torch

# cudnn.benchmark=False — переменные T после padding, autotune только мешает.
torch.backends.cudnn.benchmark = False
# TF32 для matmul — ускоряет fp32 операции на Ampere+ без потери CER.
torch.set_float32_matmul_precision("high")

# Limit torch.compile graph cache (dynamic=True can cache 50+ variants).
import torch._dynamo
torch._dynamo.config.cache_size_limit = 8


## Пути (заполните вручную)

Задай абсолютные пути под свою среду. Все `FILL_ME_IN` должны быть реальными путями к train/dev/test и директории чекпоинтов.

In [ ]:
from pathlib import Path

# DATA_ROOT was defined in the download cell above.
TRAIN_ROOT = DATA_ROOT / "data" / "train"
DEV_ROOT = DATA_ROOT / "data" / "dev"
TEST_ROOT: Path | None = DATA_ROOT / "data" / "test"
TRAIN_CSV = TRAIN_ROOT / "train.csv"
DEV_CSV = DEV_ROOT / "dev.csv"
CKPT_ROOT = Path("checkpoints") / "04_fast_conformer_bpe"

for p in (TRAIN_ROOT, DEV_ROOT, TRAIN_CSV, DEV_CSV):
    assert p.exists(), f"Path does not exist: {p}"
CKPT_ROOT.mkdir(parents=True, exist_ok=True)

print(f"train={TRAIN_ROOT}, dev={DEV_ROOT}, ckpts={CKPT_ROOT}")

MUSAN_ROOT: Path | None = Path.home() / "datasets" / "musan" / "noise"
RIR_ROOT: Path | None = Path.home() / "datasets" / "RIRS_NOISES" / "simulated_rirs"
if MUSAN_ROOT is not None and not MUSAN_ROOT.exists():
    print(f"[warn] MUSAN not found at {MUSAN_ROOT} — AddNoise disabled")
    MUSAN_ROOT = None
if RIR_ROOT is not None and not RIR_ROOT.exists():
    print(f"[warn] RIR not found at {RIR_ROOT} — RIR disabled")
    RIR_ROOT = None


## Шаг 1. Импорты

In [ ]:
import json
import random
import time

import torch
from torch.utils.data import DataLoader

from gp1.data.dataset import SpokenNumbersDataset
from gp1.data.collate import collate_fn
from gp1.data.audio_aug import AudioAugmenter
from gp1.data.audio_aug_gpu import GPUAudioAugmenter
from gp1.data.spec_aug import SpecAugmenter
from gp1.data.manifest import records_from_csv
from gp1.features.melbanks import LogMelFilterBanks
from gp1.losses.ctc import CTCLoss
from gp1.models.fast_conformer_bpe import FastConformerBPE
from gp1.text.vocab_bpe import BPEVocab, train_bpe_model
from gp1.text.denormalize import words_to_digits
from gp1.train.trainer import Trainer, TrainerConfig
from gp1.train.optim import build_adamw
from gp1.train.schedulers import build_noam
from gp1.train.checkpoint import load_checkpoint
from gp1.train.metrics import compute_cer, compute_per_speaker_cer
from gp1.decoding.greedy import greedy_decode
from gp1.types import AugConfig


## Шаг 2. Гиперпараметры (FIXED + HP_GRID)

In [ ]:
FIXED = {
    "samplerate": 16000,
    "n_fft": 512,
    "n_mels": 80,
    "hop_length": 160,
    "win_length": 400,
    "max_epochs": 80,
    "grad_accum": 1,
    "subsample_factor": 4,
    # BPE vocab size is FIXED: changing it within the trial loop would require
    # re-training SentencePiece and rebuilding CharVocab every trial.
    "bpe_vocab_size": 256,
}

HP_GRID = {
    # --- model (FastConformer — single-stage width, depth, conv kernel, heads/ff) ---
    "d_model":                   [96, 128, 160],
    "n_blocks":                  [12, 16, 20],
    "n_heads":                   [4, 8],
    "ff_ratio":                  [2, 4],
    "conv_kernel":               [9, 15, 31],
    "dropout":                   [0.05, 0.1, 0.15],

    # --- optimizer / Noam schedule ---
    "lr":                        [5e-4, 3e-4, 1e-4],
    "weight_decay":              [1e-6, 1e-4],
    "warmup_steps":              [5000, 10000, 15000],

    # --- SpecAugment (spectrogram) ---
    "specaug_freq_mask_param":   [10, 15, 20],
    "specaug_num_freq_masks":    [1, 2],
    "specaug_time_mask_param":   [20, 25, 35],
    "specaug_num_time_masks":    [2, 5, 8],
    "specaug_time_mask_max_ratio": [0.04, 0.05, 0.08],

    # --- Audio: speed & pitch (XOR — CPU) ---
    "speed_prob":                [0.5, 1.0],
    "speed_factors":             [(0.9, 1.0, 1.1), (0.85, 1.0, 1.15)],
    "pitch_prob":                [0.0, 0.3, 0.5],
    "pitch_range_semitones":     [(-2.0, 2.0), (-3.0, 3.0)],

    # --- Audio: gain (CPU) ---
    "gain_prob":                 [0.3, 0.7],
    "gain_db_range":             [(-4.0, 4.0), (-8.0, 8.0)],

    # --- Audio: GPU path (VTLP / AddNoise / RIR) ---
    "vtlp_prob":                 [0.0, 0.3, 0.5],
    "vtlp_alpha_range":          [(0.9, 1.1), (0.85, 1.15)],
    "noise_prob":                [0.0, 0.3],
    "noise_snr_db_range":        [(10.0, 20.0), (5.0, 20.0)],
    "rir_prob":                  [0.0, 0.1],
}

N_TRIALS = 6
SEED = 42
print("FIXED:", FIXED)
print("N_TRIALS:", N_TRIALS)


## Шаг 3. Загрузка записей из CSV

In [ ]:
train_records = records_from_csv(TRAIN_CSV, TRAIN_ROOT)
dev_records = records_from_csv(DEV_CSV, DEV_ROOT)
print(f"Train records: {len(train_records)}, Dev records: {len(dev_records)}")

in_domain_speakers = {r.spk_id for r in train_records}
out_of_domain_count = sum(1 for r in dev_records if r.spk_id not in in_domain_speakers)
in_domain_count = sum(1 for r in dev_records if r.spk_id in in_domain_speakers)
print(f"dev speakers: in-domain={in_domain_count} samples, out-of-domain={out_of_domain_count} samples")


## Шаг 4. BPE Vocabulary

Если файл модели не найден, обучаем SentencePiece BPE на корпусе train-транскрипций.

In [ ]:
BPE_VOCAB_SIZE = FIXED["bpe_vocab_size"]
BPE_MODEL_DIR = CKPT_ROOT.parent / "bpe_models"
BPE_MODEL_DIR.mkdir(parents=True, exist_ok=True)
BPE_MODEL_PATH = BPE_MODEL_DIR / f"bpe_{BPE_VOCAB_SIZE}.model"

if not BPE_MODEL_PATH.exists():
    # Build a plain-text corpus from train transcriptions for SP training.
    corpus_path = BPE_MODEL_DIR / "corpus.txt"
    corpus_path.write_text(
        "\n".join(r.transcription for r in train_records), encoding="utf-8"
    )
    train_bpe_model(
        corpus_path=corpus_path,
        model_prefix=str(BPE_MODEL_DIR / f"bpe_{BPE_VOCAB_SIZE}"),
        vocab_size=BPE_VOCAB_SIZE,
    )
    print(f"BPE model trained: {BPE_MODEL_PATH}")
else:
    print(f"BPE model already exists: {BPE_MODEL_PATH}")

vocab = BPEVocab(BPE_MODEL_PATH)
print(f"BPE vocab_size: {vocab.vocab_size}, blank_id: {vocab.blank_id}")


## Шаг 4.5. Предзагрузка аудио в RAM (опционально, сильно ускоряет обучение)

Загружает все train+dev файлы один раз, применяет resample до 16 kHz, и держит в RAM как тензоры. После этого `SpokenNumbersDataset.__getitem__` пропускает `sf.read` + `Resample` полностью.

Стоит ~3-5 минут один раз. Нужно ~2 GB RAM для GP1 (Colab: 12 GB, Kaggle: 29 GB — влезает с запасом).

In [ ]:
from gp1.data.dataset import preload_audio_cache

AUDIO_CACHE = preload_audio_cache(
    train_records + dev_records,
    target_samplerate=FIXED["samplerate"],
)

for k in list(AUDIO_CACHE.keys()):
    AUDIO_CACHE[k] = AUDIO_CACHE[k].contiguous().share_memory_()


## Шаг 5. HP Random Search — Training Loop

In [ ]:
import os
import random

import torch


def _worker_init(worker_id: int) -> None:
    """1 BLAS-thread per worker + per-worker RNG seed for augmenter."""
    os.environ["OMP_NUM_THREADS"] = "1"
    os.environ["MKL_NUM_THREADS"] = "1"
    torch.set_num_threads(1)

    info = torch.utils.data.get_worker_info()
    if info is None:
        return
    aug = getattr(info.dataset, "_augmenter", None)
    if aug is not None and hasattr(aug, "_rng"):
        aug._rng = random.Random(info.seed & 0xFFFFFFFF)


In [ ]:
import gc

BATCH_SIZE = 16
DL_WORKERS = 4

SEED = 42
random.seed(SEED)
trial_results = []
run_root = CKPT_ROOT / f"04_fast_conformer_bpe_{int(time.time())}"
run_root.mkdir(parents=True, exist_ok=True)

for trial in range(N_TRIALS):
    hp = {k: random.choice(v) for k, v in HP_GRID.items()}
    print(f"\n=== Trial {trial + 1}/{N_TRIALS} ===")
    print(json.dumps({**FIXED, **hp}, default=str, indent=2))

    aug_cfg = AugConfig(
        speed_factors=tuple(hp["speed_factors"]),
        speed_prob=hp["speed_prob"],
        pitch_prob=hp["pitch_prob"],
        pitch_range_semitones=tuple(hp["pitch_range_semitones"]),
        gain_prob=hp["gain_prob"],
        gain_db_range=tuple(hp["gain_db_range"]),
        seed=SEED + trial,
    )
    train_aug = AudioAugmenter(aug_cfg)
    gpu_aug = GPUAudioAugmenter(
        samplerate=FIXED["samplerate"],
        vtlp_prob=hp["vtlp_prob"],
        vtlp_alpha_range=tuple(hp["vtlp_alpha_range"]),
        noise_prob=hp["noise_prob"],
        noise_snr_db_range=tuple(hp["noise_snr_db_range"]),
        musan_root=MUSAN_ROOT,
        rir_prob=hp["rir_prob"],
        rir_root=RIR_ROOT,
    )
    spec_aug = SpecAugmenter(
        freq_mask_param=hp["specaug_freq_mask_param"],
        num_freq_masks=hp["specaug_num_freq_masks"],
        time_mask_param=hp["specaug_time_mask_param"],
        num_time_masks=hp["specaug_num_time_masks"],
        time_mask_max_ratio=hp["specaug_time_mask_max_ratio"],
        seed=SEED + trial,
    )

    train_ds = SpokenNumbersDataset(
        records=train_records,
        vocab=vocab,
        augmenter=train_aug,
        target_samplerate=FIXED["samplerate"],
        audio_cache=AUDIO_CACHE,
    )
    dev_ds = SpokenNumbersDataset(
        records=dev_records,
        vocab=vocab,
        augmenter=None,
        target_samplerate=FIXED["samplerate"],
        audio_cache=AUDIO_CACHE,
    )
    train_loader = DataLoader(
        train_ds,
        batch_size=BATCH_SIZE,
        shuffle=True,
        collate_fn=collate_fn,
        num_workers=DL_WORKERS,
        persistent_workers=True,
        pin_memory=True,
        prefetch_factor=3,
        worker_init_fn=_worker_init,
    )
    dev_loader = DataLoader(
        dev_ds,
        batch_size=BATCH_SIZE,
        shuffle=False,
        collate_fn=collate_fn,
        num_workers=DL_WORKERS,
        persistent_workers=True,
        pin_memory=True,
        prefetch_factor=3,
        worker_init_fn=_worker_init,
    )

    model = FastConformerBPE(
        vocab_size=vocab.vocab_size,
        d_model=hp["d_model"],
        n_blocks=hp["n_blocks"],
        n_heads=hp["n_heads"],
        ff_ratio=hp["ff_ratio"],
        conv_kernel=hp["conv_kernel"],
        dropout=hp["dropout"],
        subsample_factor=FIXED["subsample_factor"],
    ).to(device)
    model = torch.compile(model, mode="default", dynamic=True)

    ctc = CTCLoss(blank_id=vocab.blank_id)
    optimizer = build_adamw(
        model.parameters(),
        lr=hp["lr"],
        weight_decay=hp["weight_decay"],
    )
    scheduler = build_noam(
        optimizer,
        d_model=hp["d_model"],
        warmup_steps=hp["warmup_steps"],
    )

    trial_dir = run_root / f"trial_{trial:02d}"
    cfg = TrainerConfig(
        max_epochs=FIXED["max_epochs"],
        grad_accum=FIXED["grad_accum"],
        fp16_autocast=True,
        val_every_n_epochs=1,
        early_stop_patience=10,
        early_stop_metric="harmonic_in_out_cer",
        in_domain_speakers=in_domain_speakers,
        ckpt_dir=trial_dir,
    )
    trainer = Trainer(
        model=model,
        ctc_loss=ctc,
        optimizer=optimizer,
        scheduler=scheduler,
        train_loader=train_loader,
        val_loader=dev_loader,
        vocab=vocab,
        config=cfg,
        device=device,
        audio_cfg={
            "n_fft": FIXED["n_fft"],
            "n_mels": FIXED["n_mels"],
            "hop_length": FIXED["hop_length"],
            "win_length": FIXED["win_length"],
            "samplerate": FIXED["samplerate"],
        },
        spec_augmenter=spec_aug,
        gpu_augmenter=gpu_aug,
    )
    result = trainer.fit()
    best_ckpt = result["best_ckpt_path"]
    trial_results.append({"trial": trial, "hp": hp, **result})

    if torch.cuda.is_available():
        peak_gb = torch.cuda.max_memory_allocated() / 1e9
        print(f"trial {trial}: peak VRAM = {peak_gb:.2f} GB")

    if best_ckpt is not None:
        with open(trial_dir / "meta.json", "w") as f:
            json.dump(
                {
                    "arch": "fast_conformer_bpe",
                    "hparams": {**FIXED, **hp},
                    "val_cer": result["best_monitored"],
                    "trial": trial,
                },
                f,
                indent=2,
                default=str,
            )

    # Cleanup в конце trial.
    del trainer, model, optimizer, scheduler, ctc
    del train_loader, dev_loader, train_ds, dev_ds
    del train_aug, gpu_aug, spec_aug
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
    torch._dynamo.reset()

print("\nHP search complete.")


## Шаг 6. Сводный отчёт трайлов

In [ ]:
trial_results_sorted = sorted(trial_results, key=lambda r: r["best_monitored"])
print(f"{'trial':>5} {'best_val_cer':>12} {'lr':>8} {'dropout':>8} {'d_model':>8} {'n_blocks':>9}")
print("-" * 57)
for r in trial_results_sorted:
    hp = r["hp"]
    print(
        f"{r['trial']:>5}"
        f" {r['best_monitored']:>12.4f}"
        f" {hp['lr']:>8.5f}"
        f" {hp['dropout']:>8.4f}"
        f" {hp['d_model']:>8}"
        f" {hp['n_blocks']:>9}"
    )

## Шаг 7. Валидация лучшей модели на dev + greedy predictions

In [ ]:
best_result = trial_results_sorted[0]
best_ckpt = best_result["best_ckpt_path"]
best_hp = best_result["hp"]
print("Best trial val_cer:", best_result["best_monitored"], "ckpt:", best_ckpt)

model = FastConformerBPE(
    vocab_size=vocab.vocab_size,
    d_model=best_hp["d_model"],
    n_blocks=best_hp["n_blocks"],
    n_heads=FIXED["n_heads"],
    ff_ratio=FIXED["ff_ratio"],
    conv_kernel=best_hp["conv_kernel"],
    dropout=best_hp["dropout"],
    subsample_factor=FIXED["subsample_factor"],
).to(device)
meta = load_checkpoint(best_ckpt, model)
model.eval()

mel_extractor = LogMelFilterBanks(
    n_fft=FIXED["n_fft"],
    samplerate=FIXED["samplerate"],
    hop_length=FIXED["hop_length"],
    win_length=FIXED["win_length"],
    n_mels=FIXED["n_mels"],
).to(device)

dev_ds_eval = SpokenNumbersDataset(
    records=dev_records, vocab=vocab, augmenter=None, target_samplerate=FIXED["samplerate"]
)
dev_loader_eval = DataLoader(
    dev_ds_eval, batch_size=16, shuffle=False, collate_fn=collate_fn, num_workers=4,
        persistent_workers=True,
        pin_memory=True,
        prefetch_factor=3,
    )

predictions, refs, spks = [], [], []
with torch.no_grad():
    for batch in dev_loader_eval:
        audio = batch.audio.to(device)
        audio_lengths = batch.audio_lengths.to(device)
        mel = mel_extractor(audio)
        mel_lengths = (audio_lengths // FIXED["hop_length"] + 1).clamp(max=mel.size(-1)).long()
        out = model(mel, mel_lengths)
        hyps = greedy_decode(out.log_probs, out.output_lengths, vocab)
        predictions.extend(hyps)
        refs.extend(batch.transcriptions)
        spks.extend(batch.spk_ids)

# BPE vocab decodes piece ids -> text; then normalize to digit strings.
digit_preds = [words_to_digits(h) for h in predictions]
cer = compute_cer(refs, digit_preds)
per_spk = compute_per_speaker_cer(refs, digit_preds, spks)
print(f"Dev CER (greedy BPE): {cer:.4f}")
print("Per-speaker CER:", per_spk)

## Шаг 8. KenLM + BPE beam decoding (inline LM-pipeline)

Shallow-fusion beam search с KenLM 4-gram поверх BPE-выхода модели. Используем:

- hard lexicon (unigrams из синтетического 999k-корпуса + 42 known words);
- hotwords `тысяча / тысячи / тысяч`;
- grid search `alpha ∈ [0.3, 0.5, 0.7, 1.0, 1.3] × beta ∈ [0.0, 0.5, 1.0, 1.5]`;
- N-best + FSA-validator rescoring через `words_to_digits` (отбраковывает невалидные
  цифровые последовательности; fallback: best-beam → greedy → `"0000"`).

In [ ]:
# --- Cache dev forward pass once (reused across all beam configs) ---
dev_cache: list[dict] = []

model.eval()
with torch.no_grad():
    for batch in dev_loader_eval:
        audio = batch.audio.to(device)
        audio_lengths = batch.audio_lengths.to(device)
        mel = mel_extractor(audio)
        mel_lengths = (
            (audio_lengths // FIXED["hop_length"] + 1).clamp(max=mel.size(-1)).long()
        )
        out = model(mel, mel_lengths)
        log_probs_cpu = out.log_probs.cpu().float().numpy()
        out_lengths_cpu = out.output_lengths.cpu().numpy()
        for i in range(log_probs_cpu.shape[0]):
            L = int(out_lengths_cpu[i])
            dev_cache.append({
                "log_probs": log_probs_cpu[i, :L, :],  # [T, V]
                "out_length": L,
                "ref_digits": batch.transcriptions[i],
                "spk_id": batch.spk_ids[i],
            })
print(f"Cached dev forward pass: {len(dev_cache)} samples")

In [ ]:
# --- Build KenLM (idempotent — reuses the same lm/lm.bin as notebook 07) ---
from gp1.lm.build_corpus import build_synthetic_corpus
from gp1.lm.train_kenlm import train_kenlm

LM_DIR = Path("lm")
LM_DIR.mkdir(exist_ok=True)
corpus_path = LM_DIR / "corpus.txt"
lm_path = LM_DIR / "lm.bin"

if not corpus_path.exists():
    n_lines = build_synthetic_corpus(out_path=corpus_path, train_manifest=None)
    print(f"Corpus: {n_lines} lines -> {corpus_path}")
else:
    print(f"Corpus already exists: {corpus_path}")

if not lm_path.exists():
    train_kenlm(corpus_path, lm_path, order=4, vocab_limit_path=None)
    print(f"KenLM trained: {lm_path}")
else:
    print(f"KenLM already exists: {lm_path}")

In [ ]:
# --- BPE labels + lexicon unigrams for pyctcdecode ---
from gp1.text.denormalize import (
    _ALL_KNOWN,
    fsa_constrained_best,
    safe_words_to_digits,
)

# pyctcdecode labels: index 0 = "" (blank); 1..N = SP pieces in SP id order.
sp = vocab._sp  # SentencePieceProcessor — private API, see notebook 07 precedent.
bpe_labels: list[str] = [""] * vocab.vocab_size
for i in range(sp.get_piece_size()):
    bpe_labels[i + 1] = sp.id_to_piece(i)

# Lexicon: 42 known words + all word-forms from synthetic corpus.
lexicon_set = set(_ALL_KNOWN)
with corpus_path.open(encoding="utf-8") as f:
    for line in f:
        lexicon_set.update(line.split())
lexicon = sorted(lexicon_set)

HOTWORDS = ("тысяча", "тысячи", "тысяч")
HOTWORD_WEIGHT = 8.0
BEAM_WIDTH = 100

print(f"bpe_labels: {len(bpe_labels)}, lexicon: {len(lexicon)}, hotwords: {HOTWORDS}")

In [ ]:
# --- Beam baseline (α=0.7, β=1.0) with BPE labels ---
import pyctcdecode


def build_bpe_decoder(alpha: float, beta: float):
    """Build a pyctcdecode BPE-aware decoder with KenLM + lexicon + given (α, β)."""
    return pyctcdecode.build_ctcdecoder(
        labels=bpe_labels,
        kenlm_model_path=str(lm_path),
        unigrams=lexicon,
        alpha=alpha,
        beta=beta,
    )


def decode_all(decoder, cache, beam_width: int = BEAM_WIDTH) -> list[str]:
    hyps: list[str] = []
    for sample in cache:
        beams = decoder.decode_beams(
            sample["log_probs"],
            beam_width=beam_width,
            hotwords=list(HOTWORDS),
            hotword_weight=HOTWORD_WEIGHT,
        )
        hyps.append(beams[0][0] if beams else "")
    return hyps


refs_digits = [s["ref_digits"] for s in dev_cache]
spk_ids = [s["spk_id"] for s in dev_cache]

beam_dec_default = build_bpe_decoder(alpha=0.7, beta=1.0)
beam_hyps_default = decode_all(beam_dec_default, dev_cache)
beam_digits_default = [safe_words_to_digits(h, fallback="") for h in beam_hyps_default]
invalid_default = sum(1 for d in beam_digits_default if d == "") / len(beam_digits_default)
beam_digit_cer_default = compute_cer(refs_digits, beam_digits_default)
print(f"Beam (α=0.7, β=1.0): digit CER={beam_digit_cer_default:.4f}, invalid={invalid_default:.2%}")

In [ ]:
# --- Alpha/beta grid search on dev (fresh decoder per (α, β)) ---
import pandas as pd

ALPHAS = [0.3, 0.5, 0.7, 1.0, 1.3]
BETAS = [0.0, 0.5, 1.0, 1.5]

grid_rows = []
for alpha in ALPHAS:
    for beta in BETAS:
        dec = build_bpe_decoder(alpha, beta)
        hyps = decode_all(dec, dev_cache)
        digits = [safe_words_to_digits(h, fallback="") for h in hyps]
        invalid_pct = sum(1 for d in digits if d == "") / len(digits)
        cer = compute_cer(refs_digits, digits)
        grid_rows.append({
            "alpha": alpha,
            "beta": beta,
            "digit_cer": cer,
            "invalid_pct": invalid_pct,
        })
        print(f"α={alpha:.2f} β={beta:.2f}: digit CER={cer:.4f}, invalid={invalid_pct:.2%}")

grid_df = pd.DataFrame(grid_rows).sort_values("digit_cer").reset_index(drop=True)
best_row = grid_df.iloc[0]
BEST_ALPHA = float(best_row["alpha"])
BEST_BETA = float(best_row["beta"])
print(f"\nBest (α, β) = ({BEST_ALPHA:.2f}, {BEST_BETA:.2f}), digit CER={best_row['digit_cer']:.4f}")
grid_df

In [ ]:
# --- N-best + FSA-validator rescoring with best (α, β) ---
N_BEST = 10

best_decoder = build_bpe_decoder(BEST_ALPHA, BEST_BETA)

# Greedy baseline for fallback chain (already computed in Шаг 7).
greedy_hyps_all = predictions

fsa_hyps_digits: list[str] = []
for sample, greedy_hyp in zip(dev_cache, greedy_hyps_all):
    beams = best_decoder.decode_beams(
        sample["log_probs"],
        beam_width=BEAM_WIDTH,
        hotwords=list(HOTWORDS),
        hotword_weight=HOTWORD_WEIGHT,
    )
    top_n = beams[:N_BEST] if beams else []
    # Fallback chain: FSA-valid N-best -> best-beam digits -> greedy digits -> "0000".
    best_beam_text = beams[0][0] if beams else ""
    digits = (
        fsa_constrained_best(top_n, length_range=(4, 6))
        or safe_words_to_digits(best_beam_text, fallback="")
        or safe_words_to_digits(greedy_hyp, fallback="")
        or "0000"
    )
    fsa_hyps_digits.append(digits)

fsa_cer = compute_cer(refs_digits, fsa_hyps_digits)
fsa_fallback_pct = sum(1 for d in fsa_hyps_digits if d == "0000") / len(fsa_hyps_digits)
print(f"Beam + N-best + FSA-validator: digit CER={fsa_cer:.4f}, fallback-to-0000={fsa_fallback_pct:.2%}")

In [ ]:
# --- Comparison table: greedy / beam-default / beam-grid-best / beam+FSA ---
greedy_digits = [safe_words_to_digits(h, fallback="") for h in predictions]

beam_best_decoder = build_bpe_decoder(BEST_ALPHA, BEST_BETA)
beam_best_hyps = decode_all(beam_best_decoder, dev_cache)
beam_best_digits = [safe_words_to_digits(h, fallback="") for h in beam_best_hyps]


def _invalid_pct(digits: list[str]) -> float:
    return sum(1 for d in digits if d == "") / len(digits)


def _max_spk_cer(refs: list[str], hyps: list[str]) -> float:
    return max(compute_per_speaker_cer(refs, hyps, spk_ids).values())


methods = [
    ("greedy", predictions, greedy_digits),
    ("beam α=0.7 β=1.0", beam_hyps_default, beam_digits_default),
    (f"beam α={BEST_ALPHA} β={BEST_BETA} (grid-best)", beam_best_hyps, beam_best_digits),
    (f"beam + N-best + FSA (α={BEST_ALPHA}, β={BEST_BETA})", None, fsa_hyps_digits),
]

rows = []
for name, word_hyps, digit_hyps in methods:
    word_cer_val = (
        compute_cer(refs_digits, word_hyps) if word_hyps is not None else float("nan")
    )
    digit_cer_val = compute_cer(refs_digits, digit_hyps)
    max_spk = _max_spk_cer(refs_digits, digit_hyps)
    invalid = _invalid_pct(digit_hyps)
    rows.append({
        "method": name,
        "word_cer": word_cer_val,
        "digit_cer": digit_cer_val,
        "max_spk_cer": max_spk,
        "invalid_pct": invalid,
    })

comparison_df = pd.DataFrame(rows)
print(comparison_df.to_string(index=False))
comparison_df

In [ ]:
# --- Test submission with best beam + N-best + FSA validator -> submission_bpe.csv ---
import csv

if TEST_ROOT is not None and (TEST_ROOT / "test.csv").exists():
    from gp1.submit.inference_utils import build_test_dataloader

    test_records = records_from_csv(TEST_ROOT / "test.csv", TEST_ROOT)
    test_loader = build_test_dataloader(test_records, vocab=vocab)

    submission_decoder = build_bpe_decoder(BEST_ALPHA, BEST_BETA)

    submission_rows: list[tuple[str, str]] = []
    model.eval()
    sample_iter = iter(test_records)
    with torch.no_grad():
        for batch in test_loader:
            audio = batch.audio.to(device)
            audio_lengths = batch.audio_lengths.to(device)
            mel = mel_extractor(audio)
            mel_lengths = (
                (audio_lengths // FIXED["hop_length"] + 1).clamp(max=mel.size(-1)).long()
            )
            out = model(mel, mel_lengths)
            log_probs_cpu = out.log_probs.cpu().float().numpy()
            out_lengths_cpu = out.output_lengths.cpu().numpy()
            greedy_batch = greedy_decode(out.log_probs, out.output_lengths, vocab)
            for i in range(log_probs_cpu.shape[0]):
                L = int(out_lengths_cpu[i])
                beams = submission_decoder.decode_beams(
                    log_probs_cpu[i, :L, :],
                    beam_width=BEAM_WIDTH,
                    hotwords=list(HOTWORDS),
                    hotword_weight=HOTWORD_WEIGHT,
                )
                top_n = beams[:N_BEST] if beams else []
                # Fallback chain: FSA-valid N-best -> best-beam -> greedy -> "0000".
                best_beam_text = beams[0][0] if beams else ""
                digits = (
                    fsa_constrained_best(top_n, length_range=(4, 6))
                    or safe_words_to_digits(best_beam_text, fallback="")
                    or safe_words_to_digits(greedy_batch[i], fallback="")
                    or "0000"
                )
                rec = next(sample_iter)
                submission_rows.append((rec.audio_path.name, digits))

    submission_bpe_path = Path("notebooks") / "submission_bpe.csv"
    with submission_bpe_path.open("w", encoding="utf-8", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["filename", "transcription"])
        writer.writerows(submission_rows)
    print(f"Submission saved: {submission_bpe_path} ({len(submission_rows)} rows)")
else:
    print("No TEST_ROOT / test.csv — skipping BPE submission.")

## Шаг 9. Submission (если test данные доступны)

In [ ]:
if TEST_ROOT is not None:
    from gp1.submit.inference_utils import build_test_dataloader, write_submission

    test_records = records_from_csv(TEST_ROOT / "test.csv", TEST_ROOT)
    test_loader = build_test_dataloader(test_records, vocab=vocab)

    all_hyps = []
    model.eval()
    with torch.no_grad():
        for batch in test_loader:
            audio = batch.audio.to(device)
            audio_lengths = batch.audio_lengths.to(device)
            mel = mel_extractor(audio)
            mel_lengths = (
                (audio_lengths // FIXED["hop_length"] + 1).clamp(max=mel.size(-1)).long()
            )
            out = model(mel, mel_lengths)
            hyps = greedy_decode(out.log_probs, out.output_lengths, vocab)
            all_hyps.extend(hyps)

    # BPE vocab already decodes to text; normalize to digit strings.
    pairs = [
        (rec.audio_path.name, words_to_digits(h))
        for rec, h in zip(test_records, all_hyps)
    ]
    submission_path = run_root / "submission.csv"
    write_submission(pairs, submission_path)
    print("Submission saved:", submission_path)
else:
    print("No test_root — skipping submission.")
